In [1]:
from dotenv import load_dotenv
import os
import json
import time

load_dotenv()

True

In [2]:
import requests


def get_puuid(name, tag):
    url = "https://europe.api.riotgames.com/riot/"
    response = requests.get(f"{url}account/v1/accounts/by-riot-id/{name}/{tag}?api_key={os.getenv('APIKEY')}")
    if response.status_code == 429:
        print(f"Retrying get_puuid({name}, {tag}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_puuid(name, tag)
    return response.json()["puuid"]


puuid = get_puuid("HForGame", "euw")
matchNumber = 10
print(puuid)

uZyWLL8iUzeie3vUxD9EdYSNp329C6BbUUuGMZTxJHN2_u8x2fWieonLxAkkglFR9BqeUvu24EFjhw


In [3]:

def get_summoner_details(encrypted_summoner_id):
    url = f"https://euw1.api.riotgames.com/lol/summoner/v4/summoners/{encrypted_summoner_id}"
    params = {
        "api_key": os.getenv('APIKEY')
    }
    response = requests.get(url, params=params)
    if response.status_code == 429:
        print(f"Retrying get_summoner_details({encrypted_summoner_id}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_summoner_details(encrypted_summoner_id)
    return response.json()


In [4]:
def get_match_ids(puuid, start=0, count=20, queue=420):
    url = f"https://europe.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params = {
        "start": start,
        "count": count,
        "queue": queue,
        "api_key": os.getenv('APIKEY')
    }
    response = requests.get(url, params=params)
    if response.status_code == 429:
        print(f"Retrying get_match_ids({puuid}, {start}, {count}, {queue}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_match_ids(puuid, start, count, queue)
    return response.json()




In [5]:
def get_challenger_league(league="challengerleagues", queue="RANKED_SOLO_5x5"):
    url = f"https://euw1.api.riotgames.com/lol/league/v4/{league}/by-queue/{queue}"
    params = {
        "api_key": os.getenv('APIKEY')
    }
    response = requests.get(url, params=params)
    if response.status_code == 429:
        print(f"Retrying get_challenger_league({queue}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_challenger_league(queue)
    return response.json()
challengers = get_challenger_league()
with open("challenger.json", "w") as f:
    json.dump(challengers, f)

In [6]:
def get_match_details(match_id):
    url = f"https://europe.api.riotgames.com/lol/match/v5/matches/{match_id}"
    params = {
        "api_key": os.getenv('APIKEY')
    }
    response = requests.get(url, params=params)
    if response.status_code == 429:
        print(f"Retrying get_match_details({match_id}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_match_details(match_id)
    return response.json()

In [7]:

def get_match_timeline(match_id):
    url = f"https://europe.api.riotgames.com/lol/match/v5/matches/{match_id}/timeline"
    params = {
        "api_key": os.getenv('APIKEY')
    }
    response = requests.get(url, params=params)
    if response.status_code == 429:
        print(f"Retrying get_match_timeline({match_id}) in {response.headers['Retry-After']} seconds")
        time.sleep(int(response.headers["Retry-After"]))
        return get_match_timeline(match_id)
    return response.json()

In [8]:
teamIdToName = {
    100: "BLUE",
    200: "RED"
}

monsters = ["DRAGON", "RIFTHERALD", "BARON_NASHOR", "HORDE", "ATAKHAN"]
dragons = ["HEXTECH_DRAGON", "WATER_DRAGON", "FIRE_DRAGON", "EARTH_DRAGON", "AIR_DRAGON", "CHEMTECH_DRAGON",
           "ELDER_DRAGON"]
buildingTypes = ["TOWER_BUILDING", "INHIBITOR_BUILDING"]

extracted_data = ["level", "totalGold", "minionsKilled", "jungleMinionsKilled"]
created_data = ["kills", "deaths", "assists"]

def create_csv(match_id, match_detail, timeline, directory):
    if "info" not in timeline:
        return
    firstFrame = timeline["info"]["frames"][0]
    frame = {
        "timestamp": firstFrame["timestamp"],
        "soul": None
    }
    for match_details in match_detail["info"]["participants"]:
        frame[f"{match_details['participantId']}.champion"] = match_details["championId"]

    for participant in firstFrame["participantFrames"].values():
        for key in extracted_data:
            frame[f"{participant['participantId']}.{key}"] = participant[key]
        for key in created_data:
            frame[f"{participant['participantId']}.{key}"] = 0

    for team in teamIdToName.values():
        for monster in monsters:
            if monster == "DRAGON":
                for subtype in dragons:
                    frame[f"{team}.{subtype}.kills"] = 0
                continue
            frame[f"{team}.{monster}.kills"] = 0

    for building in buildingTypes:
        for team in teamIdToName.values():
            frame[f"{team}.{building}.kills"] = 0

    if match_detail["info"]["teams"][0]["win"]:
        frame["winner"] = "BLUE"
    else:
        frame["winner"] = "RED"

    with open(f"{directory}/{match_id}.csv", "w") as f:
        f.write(",".join(frame.keys()) + "\n")
        for frameData in timeline["info"]["frames"]:
            frame["timestamp"] = frameData["timestamp"]
            for participant in frameData["participantFrames"]:
                for key in extracted_data:
                    frame[f"{participant}.{key}"] = frameData["participantFrames"][participant][key]
            for event in frameData["events"]:
                if event["type"] == "LEVEL_UP":
                    frame[f"{event['participantId']}.level"] = event["level"]
                elif event["type"] == "CHAMPION_KILL":
                    frame[f"{event['victimId']}.deaths"] += 1
                    if event["killerId"] == 0: # "killerId" when the champion is executed.
                        continue
                    frame[f"{event['killerId']}.kills"] += 1
                    if "assistingParticipantIds" in event:
                        for assist in event["assistingParticipantIds"]:
                            frame[f"{assist}.assists"] += 1
                elif event["type"] == "ELITE_MONSTER_KILL":
                    if event["killerTeamId"] == 300: # "killerTeamId" when Baron Nashor spawns and kills Rift Herald.
                        continue
                    if event["monsterType"] == "DRAGON":
                        frame[f"{teamIdToName[event['killerTeamId']]}.{event['monsterSubType']}.kills"] += 1
                    else:
                        frame[f"{teamIdToName[event['killerTeamId']]}.{event['monsterType']}.kills"] += 1
                elif event["type"] == "DRAGON_SOUL_GIVEN":
                    frame["soul"] = event["name"]
                elif event["type"] == "BUILDING_KILL":
                    frame[f"{teamIdToName[event['teamId']]}.{event['buildingType']}.kills"] += 1
            f.write(",".join([str(i) for i in frame.values()]) + "\n")


In [9]:
import time

leagues = ["challengerleagues", "grandmasterleagues", "masterleagues"]
current_league = "masterleagues"
summonerIds = [challenger["summonerId"] for challenger in get_challenger_league(current_league)["entries"]]
size = 100
numberOfCsvDone = 0
numberOfCsvCopied = 0
numberOfSummonersDone = 0
numberOfSummoners = len(summonerIds)


def check_if_match_is_already_done(match_id):
    for league in leagues:
        if os.path.exists(f"{league}/{match_id}.csv"):
            if league != current_league:
                os.system(f"cp {league}/{match_id}.csv {current_league}/{match_id}.csv")
            return True
    return False


def create_all_csv(matchs):
    global numberOfCsvDone
    for match_id in matchs:
        if check_if_match_is_already_done(match_id):
            continue
        match = get_match_details(match_id)
        if "info" not in match or match["info"]["queueId"] != 420 or match["info"][
            "gameDuration"] < 900 or match["info"]["endOfGameResult"] != "GameComplete":
            continue
        if not match["info"]["gameVersion"].startswith("15."):
            return True
        match_timeline = get_match_timeline(match_id)
        create_csv(match_id, match, match_timeline, current_league)
        numberOfCsvDone += 1
    return False


for challengerId in summonerIds:
    try:
        challenger = get_summoner_details(challengerId)
        if "puuid" not in challenger:
            print(f"puuid not found for {challengerId}")
            continue
        puuid = challenger["puuid"]
        page = 0
        match_ids = None
        while match_ids is None or len(match_ids) >= 100:
            match_ids = get_match_ids(puuid, page * 100, 100)
            print(f"Done {numberOfCsvDone} csvs, {numberOfSummonersDone}/{numberOfSummoners} summoners done")
            if create_all_csv(match_ids):
                break
            page += 1
    except Exception as e:
        print(e)
    numberOfSummonersDone += 1

Done 0 csvs, 0/10000 summoners done
Expecting value: line 1 column 1 (char 0)
Done 1 csvs, 1/10000 summoners done
Done 1 csvs, 1/10000 summoners done
Done 1 csvs, 2/10000 summoners done
Done 1 csvs, 2/10000 summoners done
Done 1 csvs, 3/10000 summoners done
Done 1 csvs, 3/10000 summoners done
Done 1 csvs, 4/10000 summoners done
Retrying get_match_details(EUW1_7304393556) in 83 seconds
Done 4 csvs, 4/10000 summoners done
Done 4 csvs, 4/10000 summoners done
Done 4 csvs, 5/10000 summoners done
Done 13 csvs, 5/10000 summoners done
Done 13 csvs, 6/10000 summoners done
Done 22 csvs, 6/10000 summoners done
Done 22 csvs, 7/10000 summoners done
Done 22 csvs, 7/10000 summoners done
Done 22 csvs, 7/10000 summoners done
Done 22 csvs, 7/10000 summoners done
Done 22 csvs, 8/10000 summoners done
Done 34 csvs, 8/10000 summoners done
Done 34 csvs, 8/10000 summoners done
Done 34 csvs, 8/10000 summoners done
Done 34 csvs, 9/10000 summoners done
Retrying get_match_details(EUW1_7304511764) in 95 seconds
Do

KeyboardInterrupt: 

In [ ]:
import os

directory = "data"

for file in os.listdir(directory):
    match_detail = get_match_details(file.split(".")[0])
    if not match_detail["info"]["gameVersion"].startswith("15."):
        os.remove(f"data/{file}")
        print(f"Removed {file}")